# 05 (Databricks). Modele ML predictive - Consum USA (PJME)

**Versiune adaptata pentru Databricks** a notebook-ului `05_ml_consum_usa.ipynb`.

**Diferente fata de versiunea locala:**

1. **MLflow tracking** activat - fiecare model antrenat e logat automat (parametri, metrici, model serializat). Vezi rezultatele in tab-ul "Experiments" din Databricks.
2. **Paths DBFS** - datele procesate (parquet) si modelele salvate sunt in `/dbfs/FileStore/disertatie/`.
3. **Detectie GPU** - daca cluster-ul are GPU, LSTM va folosi automat (de ~10x mai rapid).
4. **%pip install** - pentru librariile care lipsesc din cluster (xgboost, prophet, holidays).

**Inainte de a rula acest notebook**, vezi `docs/DATABRICKS_SETUP.md` pentru:
- Configurare cluster (CPU vs GPU, runtime ML).
- Conectare GitHub Repo.
- Upload date in DBFS.

## Setup - instalare dependinte

In [0]:
# Instalam toate librariile necesare. Daca cluster-ul e cu ML Runtime, multe sunt deja
# instalate (will say "Requirement already satisfied"). Daca e general (16.4 LTS fara ML),
# le instaleaza pe toate.
%pip install --quiet tensorflow mlflow xgboost prophet holidays scikit-learn pyyaml

In [0]:
dbutils.library.restartPython()

## Imports si setup

In [0]:
# === Setup imports + descarcare cod din Workspace Files ===
import sys, os, time, json, base64
import urllib.request, urllib.parse
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import mlflow.sklearn
import mlflow.tensorflow

# Workspace Files in /Workspace/Users/ NU sunt accesibile prin Python open().
# Solutia: download via Databricks REST API si salvare in /tmp/ (file system normal).

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
API_TOKEN = ctx.apiToken().get()
API_URL = ctx.apiUrl().get()
NOTEBOOK_PATH = ctx.notebookPath().get()

# Repo path = parent al folder-ului notebooks (notebook e in notebooks/)
WORKSPACE_REPO_PATH = '/'.join(NOTEBOOK_PATH.split('/')[:-2])
print(f'Workspace repo path: {WORKSPACE_REPO_PATH}')

LOCAL_BASE = '/tmp/disertatie_ai_platform'

def download_workspace_file(workspace_path, local_path):
    encoded = urllib.parse.quote(workspace_path)
    url = f'{API_URL}/api/2.0/workspace/export?path={encoded}&format=SOURCE'
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {API_TOKEN}'})
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read())
    content = base64.b64decode(data['content'])
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    with open(local_path, 'wb') as f:
        f.write(content)

# Lista fisierelor de descarcat
files_to_download = [
    'src/__init__.py',
    'src/utils/__init__.py',
    'src/utils/config_loader.py',
    'src/utils/plotting.py',
    'src/data_processing/__init__.py',
    'src/data_processing/preprocessing.py',
    'src/data_processing/loader.py',
    'src/ml_models/__init__.py',
    'src/ml_models/predictors.py',
    'config.yaml',
]

print(f'Descarcare {len(files_to_download)} fisiere din Workspace in {LOCAL_BASE}...')
for f_rel in files_to_download:
    ws_full = f'{WORKSPACE_REPO_PATH}/{f_rel}'
    local_full = f'{LOCAL_BASE}/{f_rel}'
    try:
        download_workspace_file(ws_full, local_full)
        print(f'  OK: {f_rel}')
    except Exception as e:
        print(f'  FAIL: {f_rel} - {e}')

# Adaug in sys.path - acum import-urile pot gasi src/
if LOCAL_BASE not in sys.path:
    sys.path.insert(0, LOCAL_BASE)

PROJECT_ROOT = Path(LOCAL_BASE)
print(f'\nPROJECT_ROOT (local cache): {PROJECT_ROOT}')

# === Import-uri normale (codul e acum in /tmp/) ===
from src.data_processing.preprocessing import chronological_split, scale_features
from src.ml_models.predictors import (
    evaluate, train_linear, train_random_forest, train_xgboost,
    train_lstm, predict_lstm, train_prophet, predict_prophet,
    time_series_cv, tune_with_gridsearch, save_model,
    get_feature_importance, ModelResult,
)
from src.utils.config_loader import load_config
from src.utils.plotting import setup_style, PALETA

setup_style()
warnings.filterwarnings('ignore')

cfg = load_config(PROJECT_ROOT / 'config.yaml')
P_CFG = cfg['preprocessing']
print('Imports OK. Setup complet.')

## Detectie GPU

Daca cluster-ul are GPU (Standard_NC* in Azure, p2.* / g4dn.* in AWS), TensorFlow va folosi automat. Asta accelereaza LSTM cu **~10x** fata de CPU.

In [0]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs disponibile: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

if not gpus:
    print("\n[INFO] Niciun GPU detectat - LSTM va rula pe CPU.")
    print("       Pentru viteza, recomand cluster cu GPU (Single Node, Standard_NC4as_T4_v3 sau similar).")
else:
    # Activam memory growth pentru a evita ocuparea integrala a VRAM-ului
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print("\n[OK] GPU configurat pentru LSTM.")

## MLflow tracking

**Ce este MLflow?** O platforma open-source integrata in Databricks pentru tracking experimente ML.

**Ce face automat:**
- Salveaza fiecare rulare cu parametrii folositi (hyperparametri).
- Salveaza metricile obtinute (RMSE, MAE, R^2, MAPE).
- Serializeaza modelul antrenat (poti reincarca oricand).
- Iti permite sa compari runs side-by-side intr-o tabela / grafic interactiv.

**Cum vezi experimentele:** in Databricks UI, click pe iconita "Experiments" (gradina cu erlenmeyer-uri) in panoul lateral stang.

In [0]:
# Setam experimentul MLflow (creat automat la prima rulare)
EXPERIMENT_NAME = "/Users/" + dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get() + "/disertatie_consum_usa"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")

## 1. Incarcare date - detectare automata path

Notebook-ul incearca **3 locatii** in ordine pentru a gasi datele:
1. **Workspace files** in repo: `<PROJECT_ROOT>/data/processed/` - daca ai urcat datele direct in folderul Repos.
2. **DBFS legacy**: `/dbfs/FileStore/disertatie/data/processed/` - daca ai urcat prin Catalog > DBFS > Upload.
3. **Unity Catalog Volumes**: `/Volumes/<catalog>/<schema>/disertatie/data/processed/` - daca ai creat volume.

Foloseste prima locatie care contine fisierul. Daca niciuna nu functioneaza, vezi `docs/DATABRICKS_SETUP.md`.

In [0]:
# === Load data parquet ===

parquet_filename = 'consum_usa_features.parquet'
parquet_local = LOCAL_BASE + '/data/processed/' + parquet_filename

if not os.path.exists(parquet_local):
    os.makedirs(os.path.dirname(parquet_local), exist_ok=True)
    parquet_workspace = f'{WORKSPACE_REPO_PATH}/data/processed/{parquet_filename}'

    # === Strategie 1: dbutils.fs.cp cu prefix file: (cea mai sigura pentru fisiere binare) ===
    try:
        ws_url = f'file:/Workspace{parquet_workspace}'
        local_url = f'file:{parquet_local}'
        dbutils.fs.cp(ws_url, local_url)
        print(f'OK (dbutils.fs.cp): {parquet_local}')
    except Exception as e1:
        print(f'Strategie 1 (dbutils.fs.cp) esuata: {e1}')

        # === Strategie 2: shutil.copy cu path direct /Workspace/... ===
        try:
            import shutil
            src = f'/Workspace{parquet_workspace}'
            shutil.copy(src, parquet_local)
            print(f'OK (shutil.copy): {parquet_local}')
        except Exception as e2:
            print(f'Strategie 2 (shutil) esuata: {e2}')

            # === Strategie 3: cautare in DBFS / Volumes ca fallback ===
            candidates = [
                f'/dbfs/FileStore/disertatie/data/processed/{parquet_filename}',
                f'/Volumes/main/default/disertatie/data/processed/{parquet_filename}',
                f'/Volumes/workspace/default/disertatie/data/processed/{parquet_filename}',
            ]
            found = None
            for c in candidates:
                try:
                    if os.path.exists(c):
                        found = c
                        break
                except Exception:
                    continue
            if found:
                parquet_local = found
                print(f'Fisier gasit in: {parquet_local}')
            else:
                raise FileNotFoundError(
                    f'Parquet {parquet_filename} nu a fost gasit. '
                    f'Workspace path incercat: /Workspace{parquet_workspace}\n'
                    f'Solutie manuala: incarca fisierul in /dbfs/FileStore/disertatie/data/processed/'
                )
else:
    print(f'Parquet deja in cache: {parquet_local}')

df = pd.read_parquet(parquet_local)
print(f'Shape: {df.shape}')
print(f'Range: {df.index.min()} -> {df.index.max()}')
print(f'Numar features: {df.shape[1] - 1}')

## 2. Split cronologic + scaler

In [0]:
sp = chronological_split(
    df, target="PJME_MW",
    test_size=P_CFG["test_size"],
    validation_size=P_CFG["validation_size"],
)
Xt, Xv, Xs, scaler = scale_features(sp['X_train'], sp['X_val'], sp['X_test'], method='standard')
yt, yv, ys = sp['y_train'], sp['y_val'], sp['y_test']

print(f"Train: {Xt.shape}, Val: {Xv.shape}, Test: {Xs.shape}")

## 3. Regim de rulare

Pe Databricks GPU recomand `MODE = "full"` direct - hardware-ul ne permite. Daca vrei sa testezi rapid (setup, conexiuni, MLflow), ramai pe `"demo"`.

In [0]:
MODE = "full"  # "demo" sau "full"

PARAMS = {
    "demo": {
        "N_TRAIN_LSTM": 1000, "N_VAL_LSTM": 300, "N_TEST_LSTM": 500,
        "LSTM_UNITS": 16, "LSTM_EPOCHS": 3, "LSTM_BATCH": 64,
        "PROPHET_TRAIN_HOURS": 2000,
        "N_TUNE": 2000, "GRID_SPLITS": 3,
        "N_CV": 10000, "CV_SPLITS": 3,
        "RF_ESTIMATORS": 30, "RF_DEPTH": 10,
        "XGB_ESTIMATORS": 200, "XGB_DEPTH": 6,
    },
    "full": {
        "N_TRAIN_LSTM": 50000, "N_VAL_LSTM": 5000, "N_TEST_LSTM": 10000,
        "LSTM_UNITS": 64, "LSTM_EPOCHS": 30, "LSTM_BATCH": 128,
        "PROPHET_TRAIN_HOURS": 45000,
        "N_TUNE": 30000, "GRID_SPLITS": 3,
        "N_CV": 50000, "CV_SPLITS": 5,
        "RF_ESTIMATORS": 200, "RF_DEPTH": None,
        "XGB_ESTIMATORS": 300, "XGB_DEPTH": 6,
    },
}
P = PARAMS[MODE]
print(f"Mod: '{MODE}'")

## 4. Modele clasice + MLflow logging

Fiecare model antrenat va fi salvat ca un **MLflow run** separat. Astfel, dupa toate rularile, poti deschide tab-ul "Experiments" si vedea o tabela cu toate modelele ordonate dupa metrici.

In [0]:
results = []

# --- LinearRegression ---
print(">>> LinearRegression")
with mlflow.start_run(run_name="LinearRegression"):
    t = time.time()
    m = train_linear(Xt, yt)
    y_pred = m.predict(Xs)
    metrics = evaluate(ys, y_pred)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(m, "model", input_example=Xt.head())
    results.append(ModelResult(name="LinearRegression", model=m, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

# --- RandomForest ---
print(">>> RandomForest")
with mlflow.start_run(run_name="RandomForest"):
    t = time.time()
    mlflow.log_params({"n_estimators": P["RF_ESTIMATORS"], "max_depth": P["RF_DEPTH"]})
    m = train_random_forest(Xt, yt, n_estimators=P["RF_ESTIMATORS"], max_depth=P["RF_DEPTH"])
    y_pred = m.predict(Xs)
    metrics = evaluate(ys, y_pred)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(m, "model", input_example=Xt.head())
    results.append(ModelResult(name="RandomForest", model=m, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

# --- XGBoost ---
print(">>> XGBoost")
with mlflow.start_run(run_name="XGBoost"):
    t = time.time()
    mlflow.log_params({"n_estimators": P["XGB_ESTIMATORS"], "max_depth": P["XGB_DEPTH"]})
    m = train_xgboost(Xt, yt, n_estimators=P["XGB_ESTIMATORS"], max_depth=P["XGB_DEPTH"])
    y_pred = m.predict(Xs)
    metrics = evaluate(ys, y_pred)
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(m, "model", input_example=Xt.head())
    results.append(ModelResult(name="XGBoost", model=m, **metrics))
    print(f"   RMSE={metrics['rmse']:.2f}, R^2={metrics['r2']:.4f} ({time.time()-t:.1f}s)")

results.sort(key=lambda r: r.r2, reverse=True)
print("\n=== Sumar TEST ===")
for r in results:
    print(f"{r.name:<20s} RMSE={r.rmse:>8.2f} MAE={r.mae:>8.2f} R^2={r.r2:.4f} MAPE={r.mape:.2f}%")

## 5. LSTM cu MLflow + GPU acceleration

Daca ai GPU activ, LSTM va folosi automat. Vezi mesajul de mai sus la `tf.config.list_physical_devices('GPU')`. Pe GPU V100/T4 antrenarea pe 50k randuri × 30 epochs dureaza ~2-5 min (vs 30+ min CPU).

In [0]:
print(">>> LSTM cu MLflow")
with mlflow.start_run(run_name="LSTM"):
    mlflow.log_params({
        "sequence_length": 24,
        "units": P["LSTM_UNITS"],
        "epochs": P["LSTM_EPOCHS"],
        "batch_size": P["LSTM_BATCH"],
        "n_train": P["N_TRAIN_LSTM"],
    })

    N_TRAIN_LSTM = min(P["N_TRAIN_LSTM"], len(Xt))
    N_VAL_LSTM = min(P["N_VAL_LSTM"], len(Xv))
    N_TEST_LSTM = min(P["N_TEST_LSTM"], len(Xs))

    print(f"   Train: {N_TRAIN_LSTM} ore, Val: {N_VAL_LSTM} ore, Test: {N_TEST_LSTM} ore")
    t = time.time()

    lstm_bundle = train_lstm(
        X_train=Xt.iloc[-N_TRAIN_LSTM:].values,
        y_train=yt.iloc[-N_TRAIN_LSTM:].values,
        X_val=Xv.iloc[-N_VAL_LSTM:].values if len(Xv) >= N_VAL_LSTM else Xv.values,
        y_val=yv.iloc[-N_VAL_LSTM:].values if len(yv) >= N_VAL_LSTM else yv.values,
        sequence_length=24,
        units=P["LSTM_UNITS"],
        epochs=P["LSTM_EPOCHS"],
        batch_size=P["LSTM_BATCH"],
        patience=5,
        verbose=1,  # bara progres pe fiecare epoch
    )

    y_pred_lstm = predict_lstm(lstm_bundle, Xs.iloc[:N_TEST_LSTM].values)
    y_test_lstm = ys.iloc[:N_TEST_LSTM].values
    mask = ~np.isnan(y_pred_lstm)
    metrics_lstm = evaluate(y_test_lstm[mask], y_pred_lstm[mask])
    mlflow.log_metrics(metrics_lstm)

    # Salvare model Keras direct in MLflow
    import mlflow.tensorflow
    mlflow.tensorflow.log_model(lstm_bundle["model"], "model")

    results.append(ModelResult(name="LSTM", model=lstm_bundle, **metrics_lstm))
    print(f"\n   LSTM antrenat in {(time.time()-t)/60:.1f} min")
    print(f"   RMSE={metrics_lstm['rmse']:.2f}, R^2={metrics_lstm['r2']:.4f}, MAPE={metrics_lstm['mape']:.2f}%")

## 6. Prophet cu MLflow

In [0]:
print(">>> Prophet cu MLflow")
with mlflow.start_run(run_name="Prophet"):
    PROPHET_TRAIN_HOURS = min(P["PROPHET_TRAIN_HOURS"], len(yt))
    mlflow.log_param("train_hours", PROPHET_TRAIN_HOURS)

    df_train_prophet = df.iloc[len(yt) - PROPHET_TRAIN_HOURS : len(yt)]
    print(f"   Antrenare pe {PROPHET_TRAIN_HOURS} ore...")
    t = time.time()

    prophet_model = train_prophet(df_train_prophet, target="PJME_MW")

    y_pred_prophet = predict_prophet(prophet_model, sp["y_test"].index)
    metrics_prophet = evaluate(ys.values, y_pred_prophet)
    mlflow.log_metrics(metrics_prophet)

    # Prophet are propriu serializator - log_artifacts e cea mai simpla cale
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        from prophet.serialize import model_to_json
        path_json = Path(tmp) / "prophet.json"
        path_json.write_text(model_to_json(prophet_model))
        mlflow.log_artifact(str(path_json))

    results.append(ModelResult(name="Prophet", model=prophet_model, **metrics_prophet))
    print(f"   Prophet antrenat in {(time.time()-t)/60:.1f} min")
    print(f"   RMSE={metrics_prophet['rmse']:.2f}, R^2={metrics_prophet['r2']:.4f}")

## 7. GridSearchCV pentru tuning XGBoost

In [0]:
print(">>> GridSearchCV cu MLflow")
with mlflow.start_run(run_name="XGBoost_tuned_GridSearch"):
    from xgboost import XGBRegressor

    N_TUNE = min(P["N_TUNE"], len(Xt))
    Xt_tune = Xt.iloc[-N_TUNE:]
    yt_tune = yt.iloc[-N_TUNE:]

    if MODE == "demo":
        param_grid = {"n_estimators": [100, 200], "max_depth": [4, 6], "learning_rate": [0.1]}
    else:
        param_grid = {"n_estimators": [100, 200, 300], "max_depth": [4, 6, 8], "learning_rate": [0.05, 0.1]}

    n_combos = int(np.prod([len(v) for v in param_grid.values()]))
    print(f"   {n_combos} combinatii x {P['GRID_SPLITS']} folduri = {n_combos * P['GRID_SPLITS']} antrenari")
    mlflow.log_param("grid", str(param_grid))
    mlflow.log_param("n_tune", N_TUNE)

    t = time.time()
    gs = tune_with_gridsearch(
        XGBRegressor(random_state=42, n_jobs=-1),
        param_grid=param_grid,
        X_train=Xt_tune, y_train=yt_tune,
        n_splits=P["GRID_SPLITS"],
        scoring="neg_root_mean_squared_error",
        verbose=2,
    )
    print(f"   GridSearchCV terminat in {(time.time()-t)/60:.1f} min")

    mlflow.log_params({f"best_{k}": v for k, v in gs.best_params_.items()})

    y_pred_tuned = gs.best_estimator_.predict(Xs)
    metrics_tuned = evaluate(ys.values, y_pred_tuned)
    mlflow.log_metrics(metrics_tuned)
    mlflow.xgboost.log_model(gs.best_estimator_, "model")

    results.append(ModelResult(name="XGBoost_tuned", model=gs.best_estimator_, **metrics_tuned))
    print(f"   Best params: {gs.best_params_}")
    print(f"   RMSE pe TEST: {metrics_tuned['rmse']:.2f}, R^2: {metrics_tuned['r2']:.4f}")

## 8. Tabel comparativ final

In [0]:
df_comp = pd.DataFrame([r.to_dict() for r in results]).sort_values("r2", ascending=False).reset_index(drop=True)
print("=== Comparatie finala (sortat dupa R^2) ===")
print(df_comp.to_string(index=False))
print(f"\nCastigator: {df_comp.iloc[0]['model']}")
df_comp

### Plot: predictii vs real - ultima saptamana

In [0]:
last_week_idx = ys.index[-168:]
y_true_lw = ys.iloc[-168:].values

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(last_week_idx, y_true_lw, label="Real", color="black", lw=2.0, alpha=0.9)

colors = {"LinearRegression": PALETA["primary"], "RandomForest": PALETA["secondary"],
          "XGBoost": PALETA["tertiary"], "XGBoost_tuned": "#E11D48",
          "LSTM": PALETA["neutral"], "Prophet": "#9333EA"}

for r in results:
    try:
        if r.name in ("LinearRegression", "RandomForest", "XGBoost", "XGBoost_tuned"):
            y_pred_lw = r.model.predict(Xs.iloc[-168:])
        elif r.name == "Prophet":
            y_pred_lw = predict_prophet(r.model, last_week_idx)
        elif r.name == "LSTM":
            n = min(P["N_TEST_LSTM"], len(Xs))
            if n >= 168:
                y_pred_full = predict_lstm(r.model, Xs.iloc[:n].values)
                y_pred_lw = y_pred_full[n-168:n]
            else:
                continue
        ax.plot(last_week_idx, y_pred_lw, label=r.name, color=colors.get(r.name, "gray"), lw=1.2, alpha=0.8)
    except Exception as e:
        print(f"[skip] {r.name}: {e}")

ax.set_title("USA - Predictii vs Real - ultima saptamana din test (Databricks)")
ax.set_ylabel("PJME_MW")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Salvare rezultate in DBFS si export inapoi in GitHub

Modelul castigator si tabelul comparativ se salveaza in `/dbfs/FileStore/disertatie/`. Pentru a aduce in repo GitHub, fie:
1. Copiezi din DBFS in clona locala dupa ce dai pull (vezi DATABRICKS_SETUP.md sectiunea 7).
2. Lucrezi din Databricks Repos si faci commit + push direct din UI.

In [0]:
# Detectam unde sa salvam (preferinta: workspace files in repo > DBFS > Volumes)
def find_writable_output_dir(project_root: Path) -> Path:
    """Returneaza primul director scriabil pentru output (modele/rapoarte)."""
    candidates = [
        project_root,  # workspace files (in repo - se sincronizeaza prin git)
        Path("/dbfs/FileStore/disertatie"),
        Path("/Volumes/main/default/disertatie"),
        Path("/Volumes/workspace/default/disertatie"),
    ]
    for c in candidates:
        try:
            (c / "models").mkdir(parents=True, exist_ok=True)
            (c / "reports").mkdir(parents=True, exist_ok=True)
            # Test write
            test = c / "reports" / ".test_write"
            test.write_text("ok")
            test.unlink()
            return c
        except Exception:
            continue
    raise RuntimeError("Niciun director nu este scriabil pentru output.")

OUTPUT_DIR = find_writable_output_dir(PROJECT_ROOT)
print(f"Output dir: {OUTPUT_DIR}")

# Castigator (printre modelele cu API standard sklearn)
classic_results = [r for r in results if r.name not in ("LSTM", "Prophet")]
classic_results.sort(key=lambda r: r.r2, reverse=True)
winner = classic_results[0]
print(f"Castigator (clasic): {winner.name} cu R^2 = {winner.r2:.4f}")

if "XGB" in type(winner.model).__name__:
    save_path = OUTPUT_DIR / "models" / "usa_winner_xgboost_databricks.json"
    save_model(winner.model, save_path, kind="xgboost")
else:
    save_path = OUTPUT_DIR / "models" / "usa_winner_databricks.pkl"
    save_model(winner.model, save_path, kind="sklearn")
print(f"Salvat: {save_path}")

# Tabel comparativ
df_final = pd.DataFrame([r.to_dict() for r in results]).sort_values("r2", ascending=False)
df_final.insert(0, "dataset", "consum_usa")
df_final.insert(1, "platform", "databricks")
csv_path = OUTPUT_DIR / "reports" / "ml_comparison_usa_databricks.csv"
df_final.to_csv(csv_path, index=False)
print(f"Tabel salvat: {csv_path}")
df_final

## 9.5. Salvare training log detaliat (pentru analiza AI ulterioara)

Salvam toate detaliile importante intr-un fisier markdown care ajunge pe git. AI-ul va putea citi acest fisier in sesiunea urmatoare ca sa analizeze rezultatele si sa propuna optimizari.

In [0]:
import datetime

log_lines = []
log_lines.append('# Training Log - Sesiunea 1 USA pe Databricks')
log_lines.append('')
log_lines.append('Generat: ' + datetime.datetime.now().isoformat())
log_lines.append('Mod rulare: **' + str(MODE) + '**')
log_lines.append('Cluster: UTM Databricks (16.4 LTS)')
log_lines.append('Output dir: ' + str(OUTPUT_DIR))

log_lines.append('')
log_lines.append('## 1. Tabel comparativ modele')
log_lines.append('```')
log_lines.append(df_final.to_string(index=False))
log_lines.append('```')

log_lines.append('')
log_lines.append('**Castigator**: ' + str(df_final.iloc[0]['model']))
log_lines.append('**RMSE castigator**: ' + f"{df_final.iloc[0]['rmse']:.2f}")
log_lines.append('**R^2 castigator**: ' + f"{df_final.iloc[0]['r2']:.4f}")

log_lines.append('')
log_lines.append('## 2. GridSearchCV best params')
try:
    log_lines.append('```python')
    log_lines.append(str(gs.best_params_))
    log_lines.append('```')
    log_lines.append('')
    log_lines.append('Best CV RMSE: ' + f"{-gs.best_score_:.2f}")
except NameError:
    log_lines.append('GridSearchCV nu a rulat')

log_lines.append('')
log_lines.append('## 3. Feature importance top 30')
try:
    if 'XGB' in type(winner.model).__name__:
        from src.ml_models.predictors import get_feature_importance
        fi = get_feature_importance(winner.model, Xt.columns)
        log_lines.append('```')
        log_lines.append(fi.head(30).to_string(index=False))
        log_lines.append('```')
    else:
        log_lines.append('Castigatorul nu e XGBoost: ' + type(winner.model).__name__)
except Exception as e:
    log_lines.append('Eroare feature importance: ' + str(e))

log_lines.append('')
log_lines.append('## 4. LSTM training history')
try:
    if hasattr(lstm_bundle.get('model'), 'history'):
        hist = lstm_bundle['model'].history.history
        log_lines.append('Epochs antrenate: ' + str(len(hist.get('loss', []))))
        log_lines.append('```')
        log_lines.append('Epoch    Loss        Val Loss    MAE')
        for i in range(len(hist.get('loss', []))):
            ml = hist['loss'][i]
            vl = hist.get('val_loss', [None]*len(hist['loss']))[i] if 'val_loss' in hist else None
            mae_v = hist.get('mae', [None]*len(hist['loss']))[i] if 'mae' in hist else None
            vl_str = f'{vl:.4f}' if isinstance(vl, float) else 'N/A'
            mae_str = f'{mae_v:.4f}' if isinstance(mae_v, float) else 'N/A'
            log_lines.append(f'{i+1:<9}{ml:<12.4f}{vl_str:<12}{mae_str:<12}')
        log_lines.append('```')
    else:
        log_lines.append('LSTM history nu este disponibil')
except Exception as e:
    log_lines.append('Eroare LSTM history: ' + str(e))

log_lines.append('')
log_lines.append('## 5. Setari utilizate')
try:
    log_lines.append('```python')
    for k, v in P.items():
        log_lines.append(str(k) + ': ' + str(v))
    log_lines.append('```')
except NameError:
    log_lines.append('PARAMS nu disponibil')

log_lines.append('')
log_lines.append('## 6. Date despre dataset')
log_lines.append('- Total randuri: ' + str(len(df)))
log_lines.append('- Numar features: ' + str(df.shape[1] - 1))
log_lines.append('- Train: ' + str(len(sp['X_train'])))
log_lines.append('- Val: ' + str(len(sp['X_val'])))
log_lines.append('- Test: ' + str(len(sp['X_test'])))

log_path = OUTPUT_DIR / 'reports' / 'training_log_usa_databricks.md'
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text(chr(10).join(log_lines), encoding='utf-8')
print('Training log salvat: ' + str(log_path))
print('Continut: ' + str(len(log_lines)) + ' linii')

print('')
print('=== Preview primele 30 linii ===')
print(chr(10).join(log_lines[:30]))

In [0]:
# Copiez rezultatele din /tmp/ inapoi in workspace files (pentru git)
import shutil

LOCAL_OUTPUT = '/tmp/disertatie_ai_platform'
WORKSPACE_BASE = '/Workspace' + WORKSPACE_REPO_PATH

files_to_copy = [
    'models/usa_winner_xgboost_databricks.json',
    'models/usa_winner_databricks.pkl',
    'reports/ml_comparison_usa_databricks.csv',
    'reports/training_log_usa_databricks.md',
]

for f_rel in files_to_copy:
    src = LOCAL_OUTPUT + '/' + f_rel
    dst = WORKSPACE_BASE + '/' + f_rel
    
    if not os.path.exists(src):
        print('SKIP (nu exista in /tmp): ' + f_rel)
        continue
    
    try:
        # dbutils.fs.cp cu prefix file: pentru workspace files
        dbutils.fs.cp('file:' + src, 'file:' + dst)
        print('OK: ' + f_rel + ' -> workspace')
    except Exception as e:
        print('FAIL dbutils: ' + f_rel + ': ' + str(e))
        try:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
            print('OK shutil: ' + f_rel + ' -> workspace')
        except Exception as e2:
            print('FAIL shutil: ' + f_rel + ': ' + str(e2))

print('')
print('Done! Verifica acum dialogul Git - ar trebui sa apara fisierele noi.')

## 10. Concluzii Sesiunea 1 (Databricks)

**Avantaje observate fata de rularea locala:**

| Aspect | Local CPU | Databricks |
|---|---|---|
| LSTM 50k × 30 epochs | 30-60 min | ~2-5 min (GPU) |
| GridSearchCV 18 combos | 15-30 min | ~3-8 min |
| Tracking experimente | manual (CSV) | automat (MLflow UI) |
| Vizibilitate profesoara | trimite fisier | partajat link UI |

**Pas urmator:**
- Vezi tab-ul "Experiments" in Databricks UI pentru toate run-urile inregistrate.
- Compara automat parametri si metrici.
- Pentru lucrare: poti exporta din MLflow tabele si grafice direct in capitolul de implementare.